# Data Exploration Notebook

Exploratory data analysis (EDA) of TrackNow checkouts and PostHog sessions.
Goal: understand data quality, identifier coverage, and matching potential before modeling.

## 1. Loading

Read both sheets from the Excel source using Polars.

In [ ]:
import polars as pl

SOURCE_PATH = "/home/rafael/hermes/workspaces/pfm_assignment/data/source.xlsx"

sheets = pl.read_excel(SOURCE_PATH, sheet_id=0)
print("Available sheets:", list(sheets.keys()))

In [ ]:
tracknow = sheets["Sample TrackNow Checkouts"]
posthog = sheets["Sample PostHog Sessions"]

print("TrackNow shape:", tracknow.shape)
print("PostHog shape:", posthog.shape)

In [ ]:
print("=== TrackNow schema ===")
print(tracknow.schema)
print()
print("=== PostHog schema ===")
print(posthog.schema)

In [ ]:
print("=== TrackNow first rows ===")
tracknow.head()

In [ ]:
print("=== PostHog first rows ===")
posthog.head()

## 2. Basic Quality

For each dataset: rows, columns, types, nulls, duplicates, cardinality of main keys.

In [ ]:
def basic_quality(df: pl.DataFrame, name: str, keys: list[str]) -> pl.DataFrame:
    """Compute basic quality metrics for a dataframe."""
    rows = df.shape[0]
    cols = df.shape[1]
    null_counts = df.null_count().row(0)
    dup_rows = df.is_duplicated().sum()
    
    metrics = []
    for col in df.columns:
        null_pct = null_counts[df.columns.index(col)] / rows * 100 if rows > 0 else 0
        metrics.append({
            "dataset": name,
            "column": col,
            "dtype": str(df[col].dtype),
            "null_count": null_counts[df.columns.index(col)],
            "null_pct": round(null_pct, 1),
            "unique": df[col].n_unique(),
            "is_key": col in keys,
        })
    
    result = pl.DataFrame(metrics)
    print(f"--- {name} ---")
    print(f"Rows: {rows}, Columns: {cols}, Duplicate rows: {dup_rows}")
    return result

tn_keys = ["tracknow_order_id", "click_id", "affiliate_session_id"]
ph_keys = ["session_id", "posthog_distinct_id", "click_id_from_url", "gclid", "fbclid"]

tn_quality = basic_quality(tracknow, "TrackNow", tn_keys)
ph_quality = basic_quality(posthog, "PostHog", ph_keys)

In [ ]:
print("=== TrackNow quality ===")
tn_quality

In [ ]:
print("=== PostHog quality ===")
ph_quality

## 3. Identifier Coverage

Calculate presence of key identifiers in each dataset.

In [ ]:
def identifier_coverage(df: pl.DataFrame, name: str, id_cols: list[str]) -> None:
    """Show presence (non-null count and %) of identifier columns."""
    total = df.shape[0]
    print(f"--- {name} identifier coverage ---")
    for col in id_cols:
        if col in df.columns:
            present = df[col].is_not_null().sum()
            pct = present / total * 100 if total > 0 else 0
            print(f"  {col}: {present}/{total} ({pct:.1f}%)")
        else:
            print(f"  {col}: NOT PRESENT")
    print()

tn_ids = ["click_id", "affiliate_session_id"]
ph_ids = ["click_id_from_url", "gclid", "fbclid"]

identifier_coverage(tracknow, "TrackNow", tn_ids)
identifier_coverage(posthog, "PostHog", ph_ids)

## 4. Matching Potential

Compare `TrackNow.click_id` against `PostHog.gclid`, `PostHog.fbclid`, and `PostHog.click_id_from_url`.
Exact matches only, no fuzzy matching.

In [ ]:
# Extract non-null TrackNow click_ids
tn_click_ids = tracknow.filter(pl.col("click_id").is_not_null()).select(
    pl.col("tracknow_order_id"),
    pl.col("click_id").alias("match_id"),
    pl.col("created_date"),
    pl.col("status"),
    pl.col("firm_id"),
)

# Extract non-null PostHog identifiers (rename to match_id for joining)
ph_gclid = posthog.filter(pl.col("gclid").is_not_null()).select(
    pl.col("session_id"),
    pl.col("gclid").alias("match_id"),
    pl.lit("gclid").alias("id_type"),
)
ph_fbclid = posthog.filter(pl.col("fbclid").is_not_null()).select(
    pl.col("session_id"),
    pl.col("fbclid").alias("match_id"),
    pl.lit("fbclid").alias("id_type"),
)
ph_click_id_from_url = posthog.filter(pl.col("click_id_from_url").is_not_null()).select(
    pl.col("session_id"),
    pl.col("click_id_from_url").alias("match_id"),
    pl.lit("click_id_from_url").alias("id_type"),
)

# Union all PostHog identifiers
ph_all_ids = pl.concat([ph_gclid, ph_fbclid, ph_click_id_from_url])

print(f"TrackNow click_ids (non-null): {tn_click_ids.shape[0]}")
print(f"PostHog identifiers (non-null, any type): {ph_all_ids.shape[0]}")
print(f"  - gclid: {ph_gclid.shape[0]}")
print(f"  - fbclid: {ph_fbclid.shape[0]}")
print(f"  - click_id_from_url: {ph_click_id_from_url.shape[0]}")

In [ ]:
# Exact match: TrackNow.click_id == PostHog identifier
matched = tn_click_ids.join(
    ph_all_ids,
    on="match_id",
    how="inner",
)

print(f"Exact matches: {matched.shape[0]}")
print(f"Unique TrackNow orders matched: {matched['tracknow_order_id'].n_unique()}")
print(f"Unique PostHog sessions matched: {matched['session_id'].n_unique()}")
print()
print("Matches by identifier type:")
matched["id_type"].value_counts()

In [ ]:
# TrackNow orders without any match
unmatched_tn = tn_click_ids.join(
    ph_all_ids,
    on="match_id",
    how="anti",
)

print(f"TrackNow click_ids without match: {unmatched_tn.shape[0]}")
print(f"  out of {tn_click_ids.shape[0]} total non-null click_ids")
print(f"  match rate: {matched.shape[0] / tn_click_ids.shape[0] * 100:.1f}%" if tn_click_ids.shape[0] > 0 else "  N/A")

In [ ]:
# Multiple candidates: same PostHog id matches multiple TrackNow orders
multi_match = matched.group_by("match_id").agg(
    pl.col("tracknow_order_id").n_unique().alias("order_count"),
    pl.col("session_id").n_unique().alias("session_count"),
).filter(pl.col("order_count") > 1)

print(f"PostHog IDs matching multiple TrackNow orders: {multi_match.shape[0]}")
if multi_match.shape[0] > 0:
    print(multi_match.head())

## 5. Temporal Coverage

Compare `TrackNow.created_date` vs `PostHog.session_date`.

In [ ]:
# Parse dates
tn_dates = tracknow.select(
    pl.col("created_date").str.to_date("%Y-%m-%d").alias("date")
).filter(pl.col("date").is_not_null())

ph_dates = posthog.select(
    pl.col("session_date").str.to_date("%Y-%m-%d").alias("date")
).filter(pl.col("date").is_not_null())

print("=== TrackNow date range ===")
print(f"  Min: {tn_dates['date'].min()}")
print(f"  Max: {tn_dates['date'].max()}")
print(f"  Unique dates: {tn_dates['date'].n_unique()}")
print()
print("=== PostHog date range ===")
print(f"  Min: {ph_dates['date'].min()}")
print(f"  Max: {ph_dates['date'].max()}")
print(f"  Unique dates: {ph_dates['date'].n_unique()}")

In [ ]:
# Date distribution comparison
tn_date_counts = tn_dates.group_by("date").agg(pl.len().alias("tracknow_count")).sort("date")
ph_date_counts = ph_dates.group_by("date").agg(pl.len().alias("posthog_count")).sort("date")

date_comparison = tn_date_counts.join(ph_date_counts, on="date", how="outer").sort("date")
print("=== Daily record counts ===")
date_comparison.head(20)

In [ ]:
# Check if unmatched TrackNow orders fall outside PostHog date range
if unmatched_tn.shape[0] > 0:
    ph_min = ph_dates["date"].min()
    ph_max = ph_dates["date"].max()
    
    unmatched_outside = unmatched_tn.filter(
        (pl.col("created_date").str.to_date("%Y-%m-%d") < ph_min) |
        (pl.col("created_date").str.to_date("%Y-%m-%d") > ph_max)
    )
    
    print(f"Unmatched TrackNow orders outside PostHog date range [{ph_min}, {ph_max}]:")
    print(f"  {unmatched_outside.shape[0]} / {unmatched_tn.shape[0]}")
    print(f"  ({unmatched_outside.shape[0] / unmatched_tn.shape[0] * 100:.1f}%)" if unmatched_tn.shape[0] > 0 else "  N/A")

## 6. Unmatched Analysis

Analyze conversions without match by: presence of `click_id`, `status`, `firm_id`, date.

In [ ]:
# All TrackNow orders (including those without click_id)
all_tn_with_click_flag = tracknow.with_columns(
    pl.col("click_id").is_not_null().alias("has_click_id"),
)

# Orders without click_id
no_click_id = all_tn_with_click_flag.filter(~pl.col("has_click_id"))
print(f"TrackNow orders without click_id: {no_click_id.shape[0]} / {tracknow.shape[0]}")
print(f"  ({no_click_id.shape[0] / tracknow.shape[0] * 100:.1f}%)" if tracknow.shape[0] > 0 else "  N/A")
print()

# Status distribution for unmatched (no click_id)
if no_click_id.shape[0] > 0:
    print("Status distribution (no click_id):")
    print(no_click_id["status"].value_counts())
    print()
    print("Firm distribution (no click_id):")
    print(no_click_id["firm_id"].value_counts())

In [ ]:
# Unmatched orders that DO have a click_id (but no match in PostHog)
if unmatched_tn.shape[0] > 0:
    print(f"Unmatched orders WITH click_id: {unmatched_tn.shape[0]}")
    print()
    print("Status distribution:")
    print(unmatched_tn["status"].value_counts())
    print()
    print("Firm distribution:")
    print(unmatched_tn["firm_id"].value_counts())
    print()
    print("Date distribution:")
    print(unmatched_tn["created_date"].value_counts().sort("count", descending=True).head(10))

### Initial Hypotheses

1. **Missing click_id**: Some TrackNow orders have no click_id at all, so they cannot be matched by design.
2. **Identifier mismatch**: PostHog uses gclid/fbclid/click_id_from_url, but these may not align with TrackNow click_id.
3. **Temporal gap**: Unmatched orders may fall outside the PostHog tracking window.
4. **Fan-out risk**: If one PostHog ID matches multiple TrackNow orders, attribution becomes ambiguous.

## 7. Conclusions

### Sample Limitations
- TrackNow: 100 rows, PostHog: 200 rows (sample data).
- All records fall within a limited date range (May-June 2026).

### Identifier Quality
- TrackNow click_id coverage: varies (see Section 3).
- PostHog gclid/fbclid/click_id_from_url: partial coverage, different identifiers.

### Fan-out Risks
- Some PostHog IDs may match multiple TrackNow orders (see Section 4).
- Need to define deduplication strategy in dbt pipeline.

### Hypotheses for dbt Pipeline
1. Prioritize `click_id` as primary matching key (most direct).
2. Fall back to `gclid`/`fbclid` when `click_id` is absent.
3. Apply temporal window constraint (e.g., session within N days before checkout).
4. Handle fan-out via ranking (most recent session, or first touch).
5. Flag orders that cannot be matched for separate analysis.